# 🎙️ Sutta TTS Training Control Panel
**Version:** 12.0 | **Author:** SuttaPlayer

This notebook provides the documented commands to manage your Piper TTS training session. It is designed for the **Colab Free Tier** with a focus on:
1. **Persistence:** Using `tmux` to keep training running if a cell disconnects.
2. **Safety:** Smart pruning to prevent `/content` disk crashes.
3. **Recovery:** Easy restore commands for your environment.

---

## ⚠️ Before You Start
- **Runtime:** Ensure you are using a **GPU Runtime** (preferably T4 or A100 if available). Go to `Runtime > Change runtime type > Hardware accelerator: GPU`.
- **Drive:** Ensure Google Drive is mounted in the first cell below.
- **Terminal:** This panel assumes you have access to the Colab Terminal (View > Show terminal).

## 🚀 Step 1: Environment Setup & Mounting
Run this cell once to mount Drive and verify your environment.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

# Verify paths exist
paths = [
    "/content/drive/MyDrive/sutta-tts-model-training",
    "/content/drive/MyDrive/piper_training",
    "/content/piper_cache"
]

for p in paths:
    if os.path.exists(p):
        print(f"✅ Found: {p}")
    else:
        print(f"⚠️  Missing: {p} (Creating directory...)")
        os.makedirs(p, exist_ok=True)

print("\n✅ Environment Ready. Proceed to Step 2.")

## 🚀 Step 2: Launch Training in Tmux (Background)
**Why Tmux?** If a Colab cell disconnects, the training process stops. `tmux` keeps it running in a separate terminal session. You can detach from it and come back later.

**Action:** Copy the command below into the **Colab Terminal** (not this cell).

### 📋 Copy This Command to Terminal

Open the **Colab Terminal** (View > Show terminal) and paste:

```bash
# 1. Ensure the manager script exists (if not, re-upload it)
ls /content/sutta-training-manager.ts

# 2. Launch training in background (Tmux)
deno run --allow-all /content/sutta-training-manager.ts --train
```

**What happens:**
- It creates a tmux session named `piper_train`.
- Training starts automatically.
- You can safely close the cell and the training continues.

## 🔍 Step 3: Manage the Session
Here are the commands to attach, detach, or check the session.

### 📋 Attach to Training Session (to see logs)
Paste this in the **Terminal**:
```bash
tmux attach -t piper_train
```
*Press `Ctrl+B`, then `D` to detach safely.*

### 📋 Run Monitor (Sync & Prune)
Paste this in the **Terminal** to sync checkpoints to Drive and prune local disk:
```bash
deno run --allow-all /content/sutta-training-manager.ts --monitor
```
*This keeps your local disk from filling up.*

## 📊 Step 4: Sync UAT Metrics to Google Sheets
Run this cell to push your latest validation metrics to your Drive CSV and optionally to Sheets.

In [ ]:
import os
from google.colab import drive, sheets
import pandas as pd

drive.mount('/content/drive')

# Paths
CSV_PATH = "./uat_metrics.csv"
DRIVE_CSV = "/content/drive/MyDrive/piper_training/uat_metrics.csv"
SHEET_ID = "YOUR_SHEET_ID_HERE"  # ⚠️ REPLACE THIS WITH YOUR SHEET ID

if os.path.exists(CSV_PATH):
    print("📊 Reading UAT metrics...")
    df = pd.read_csv(CSV_PATH)
    df.to_csv(DRIVE_CSV, index=False)
    print(f"✅ Saved to Drive: {DRIVE_CSV}")
    
    if SHEET_ID != "YOUR_SHEET_ID_HERE":
        print("📤 Appending to Google Sheets...")
        data = df.values.tolist()
        try:
            sheets.values_append(
                sheet_id=SHEET_ID,
                range="A1",
                value_input_option="USER_ENTERED",
                body={"values": data}
            )
            print("🎉 Successfully synced to Google Sheets!")
        except Exception as e:
            print(f"❌ Sheets error: {e}")
    else:
        print("⚠️  No Sheet ID provided. Skipping Sheets append.")
else:
    print("⚠️  No UAT CSV found yet. Training must have run at least one validation.")

## 🔄 Step 5: Recovery & Restore
If you need to reset your environment or restore from a backup.

### 📋 Restore Pip Environment
Run this in the **Terminal** if you need to reinstall dependencies:
```bash
deno run --allow-all /content/sutta-training-manager.ts --pip-restore
```

### 📋 Restore Piper Cache
Run this in the **Terminal** to restore the phoneme cache:
```bash
deno run --allow-all /content/sutta-training-manager.ts --cache-restore
```

## 💾 Step 6: Disk Space Check (Safety)
Run this in the **Terminal** to check your current disk usage before training crashes:

In [ ]:
!df -h /content